# Lecture 35 Worksheet: BERT, GPT, T5, Pre-training and Cost
**NAVTTC / NIAI AI/ML Trainer Cohort** — runs in Google Colab or local Jupyter (PyTorch, CPU is enough, about 10 minutes in total). Fill every `None` / `# YOUR CODE` area; check cells print `OK` or tell you what to fix.

We cannot download BERT or GPT here, so we build **tiny** versions of the three families on our own Urdu–English corpus and measure what the notes (Pages 12 to 15) only describe: what each family sees, what pre-training gives you, how temperature changes generation, how the cost of attention grows with length, what a KV cache saves, and what int8 quantisation costs. Every number you see comes from these cells.

In [ ]:
import os, math, json, time, random
os.environ["OMP_NUM_THREADS"] = "2"
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
torch.set_num_threads(2); np.set_printoptions(precision=3, suppress=True)
torch.manual_seed(0); np.random.seed(0); random.seed(0)
R = {}          # results we collect, printed as one JSON line at the very end
def check(name, ok, hint=""):
    print(("OK   " if ok else "FIX  ") + name + ("" if ok else "  -> " + hint))
print("torch", torch.__version__)
try:
    import sacrebleu
except ImportError:
    import subprocess, sys; subprocess.run([sys.executable, "-m", "pip", "install", "-q", "sacrebleu"]); import sacrebleu

## Provided: the building blocks from Lectures 32 and 33

In [ ]:
def attention(Q, K, V, mask=None):
    d_k = Q.shape[-1]
    scores = Q @ K.transpose(-2, -1) / math.sqrt(d_k)
    if mask is not None: scores = scores.masked_fill(~mask, float("-inf"))
    w = F.softmax(scores, dim=-1)
    return w @ V, w
def split_heads(x, h):
    B, n, d = x.shape
    return x.view(B, n, h, d // h).transpose(1, 2)
def merge_heads(x):
    B, h, n, dk = x.shape
    return x.transpose(1, 2).reshape(B, n, h * dk)
class MultiHeadAttention(nn.Module):
    def __init__(self, d, h):
        super().__init__()
        assert d % h == 0
        self.d, self.h = d, h
        self.W_q = nn.Linear(d, d, bias=False); self.W_k = nn.Linear(d, d, bias=False)
        self.W_v = nn.Linear(d, d, bias=False); self.W_o = nn.Linear(d, d, bias=False)
    def forward(self, q_in, kv_in, mask=None):
        Q, K, V = split_heads(self.W_q(q_in), self.h), split_heads(self.W_k(kv_in), self.h), split_heads(self.W_v(kv_in), self.h)
        heads, w = attention(Q, K, V, mask)
        return self.W_o(merge_heads(heads)), w
class FeedForward(nn.Module):
    def __init__(self, d, d_ff):
        super().__init__(); self.l1 = nn.Linear(d, d_ff); self.l2 = nn.Linear(d_ff, d)
    def forward(self, x): return self.l2(F.relu(self.l1(x)))
class EncoderBlock(nn.Module):
    def __init__(self, d, h, d_ff, p=0.1):
        super().__init__()
        self.attn, self.ffn = MultiHeadAttention(d, h), FeedForward(d, d_ff)
        self.ln1, self.ln2, self.drop = nn.LayerNorm(d), nn.LayerNorm(d), nn.Dropout(p)
    def forward(self, x, mask=None):
        x = self.ln1(x + self.drop(self.attn(x, x, mask)[0]))
        return self.ln2(x + self.drop(self.ffn(x)))
def sinusoidal_pe(max_len, d_model):
    pe = torch.zeros(max_len, d_model)
    pos = torch.arange(max_len).unsqueeze(1).float(); div = torch.pow(10000.0, torch.arange(0, d_model, 2).float() / d_model)
    pe[:, 0::2] = torch.sin(pos / div); pe[:, 1::2] = torch.cos(pos / div); return pe
print("ok")

## Task 1: What can each family see?

All three families are built from the same blocks. They differ in the **mask**:

* **encoder-only (BERT)**: every word sees every word.
* **decoder-only (GPT)**: every word sees itself and the words BEFORE it.
* **encoder-decoder (T5)**: the encoder sees everything, the decoder is causal over its own words and looks at ALL source words through cross-attention.

Write `family_masks(kind, n_src, n_tgt)`. It returns a dict of boolean matrices (`True` = may look). Keys: `"encoder-only"` → `{"self": (n_src, n_src)}`, `"decoder-only"` → `{"self": (n_src, n_src)}` (n_tgt ignored), `"encoder-decoder"` → `{"encoder": (n_src, n_src), "decoder": (n_tgt, n_tgt), "cross": (n_tgt, n_src)}`.

In [ ]:
def family_masks(kind, n_src, n_tgt=0):
    if kind == "encoder-only":
        return {"self": None}                       # YOUR CODE
    if kind == "decoder-only":
        return {"self": None}                       # YOUR CODE
    if kind == "encoder-decoder":
        return {"encoder": None, "decoder": None, "cross": None}   # YOUR CODE
    raise ValueError(kind)

In [ ]:
bert, gpt, t5 = family_masks("encoder-only", 5), family_masks("decoder-only", 5), family_masks("encoder-decoder", 5, 4)
check("BERT: all True", bool(bert["self"].all()) and tuple(bert["self"].shape) == (5, 5))
check("GPT: lower triangle", torch.equal(gpt["self"], torch.tril(torch.ones(5, 5, dtype=torch.bool))))
check("T5 encoder / decoder / cross shapes", tuple(t5["encoder"].shape) == (5, 5) and tuple(t5["decoder"].shape) == (4, 4) and tuple(t5["cross"].shape) == (4, 5))
check("T5 decoder is causal, cross is open", bool(t5["decoder"][0, 1]) is False and bool(t5["cross"].all()))
visible = {"BERT": int(bert["self"].sum()), "GPT": int(gpt["self"].sum()), "T5 decoder": int(t5["decoder"].sum())}
print("number of (word, word) pairs that may interact, n=5:", visible)

## Our corpus
The same Urdu→English toy corpus as Lectures 30 to 34. English sentences are used to train the tiny GPT; Urdu sentences are used to pre-train the tiny BERT-style encoder; translation pairs are used for T5-style fine-tuning.

In [ ]:
import random

TRANSLIT = {"لڑکا":"larka","لڑکی":"larki","استاد":"ustaad","بچہ":"bacha","علی":"Ali","سارہ":"Sara","باپ":"baap","ماں":"maan",
 "ڈاکٹر":"doctor","نرس":"nurse","پڑھتا":"parhta","پڑھتی":"parhti","کھاتا":"khata","کھاتی":"khati","پیتا":"peeta","پیتی":"peeti",
 "دیکھتا":"dekhta","دیکھتی":"dekhti","لکھتا":"likhta","لکھتی":"likhti","ہے":"hai","اور":"aur","کتاب":"kitab","اخبار":"akhbar",
 "خط":"khat","روٹی":"roti","سیب":"saib","چاول":"chawal","چائے":"chai","پانی":"pani","دودھ":"doodh","فلم":"film","گاڑی":"gaari",
 "پرندہ":"parinda","کہانی":"kahani","نام":"naam","گھر":"ghar","سکول":"school","میں":"mein","<EOS>":"<EOS>"}
def build_pairs(n_two=700, n_three=700, seed=0):
    r = random.Random(seed)
    subj = [("لڑکا","the boy","m"),("لڑکی","the girl","f"),("استاد","the teacher","m"),("بچہ","the child","m"),
            ("علی","Ali","m"),("سارہ","Sara","f"),("باپ","the father","m"),("ماں","the mother","f"),
            ("ڈاکٹر","the doctor","m"),("نرس","the nurse","f")]
    verbs = {  # (urdu masc, urdu fem, english 3rd person)
      "read":("پڑھتا","پڑھتی","reads",[("کتاب","the book"),("اخبار","the newspaper"),("خط","the letter")]),
      "eat": ("کھاتا","کھاتی","eats",[("روٹی","bread"),("سیب","the apple"),("چاول","rice")]),
      "drink":("پیتا","پیتی","drinks",[("چائے","tea"),("پانی","water"),("دودھ","milk")]),
      "see": ("دیکھتا","دیکھتی","sees",[("فلم","the movie"),("گاڑی","the car"),("پرندہ","the bird")]),
      "write":("لکھتا","لکھتی","writes",[("کہانی","the story"),("نام","the name"),("خط","the letter")]),
    }
    places = [("", ""), ("گھر میں","at home"), ("سکول میں","at school")]
    simple = []
    for su,se,g in subj:
        for vk,(vm,vf,ve,objs) in verbs.items():
            for ou,oe in objs:
                for pu,pe in places:
                    u = f"{su} {pu} {ou} {vm if g=='m' else vf} ہے".replace("  "," ")
                    e = f"{se[0].upper()+se[1:]} {ve} {oe} {pe}".strip()
                    simple.append((u,e))
    simple = sorted(set(simple))
    def join(parts):
        u = " اور ".join(p[0] for p in parts)
        es = [parts[0][1]] + [(p[1][0].lower() + p[1][1:]) if p[1].startswith("The ") else p[1] for p in parts[1:]]
        return (u, " and ".join(es))
    comp = set()
    while len(comp) < n_two:
        comp.add(join(r.sample(simple, 2)))
    while len(comp) < n_two + n_three:
        comp.add(join(r.sample(simple, 3)))
    pairs = simple + sorted(comp)
    r.shuffle(pairs)
    return pairs

pairs = build_pairs(); random.Random(42).shuffle(pairs)
split = int(0.85 * len(pairs)); train, valid = pairs[:split], pairs[split:]
valid_src = {a for a, b in valid}
big = [p for p in build_pairs(4000, 4000, 0) if p[0] not in valid_src]; random.Random(7).shuffle(big)
print(len(train), "train /", len(valid), "validation pairs;", len(big), "extra unlabelled-style pairs (validation sentences removed)")
PAD, BOS, EOS = 0, 1, 2
def make_vocab(sents, extra=()):
    v = ["<pad>", "<start>", "<end>"] + list(extra) + sorted({w for s in sents for w in s.split()}); return {w: i for i, w in enumerate(v)}
src_v = make_vocab([a for a, b in pairs] + [a for a, b in big], extra=["<mask>"]); tgt_v = make_vocab([b for a, b in pairs] + [b for a, b in big])
inv_t = {i: w for w, i in tgt_v.items()}; MASK = src_v["<mask>"]
def pad_batch(seqs):
    L_ = max(map(len, seqs)); x = torch.zeros(len(seqs), L_, dtype=torch.long)
    for i, s in enumerate(seqs): x[i, :len(s)] = torch.tensor(s)
    return x
def encode_pairs(ps):
    return (pad_batch([[src_v[w] for w in a.split()] for a, b in ps]), pad_batch([[BOS] + [tgt_v[w] for w in b.split()] for a, b in ps]),
            pad_batch([[tgt_v[w] for w in b.split()] + [EOS] for a, b in ps]))
print("vocab (Urdu incl. <mask> / English):", len(src_v), len(tgt_v))

## Task 2: BERT's training data — masked language modelling

Write `mask_tokens(ids, p, g)` for a batch `ids` of shape `(B, T)` (0 = padding). Pick each REAL token with probability `p` (use `torch.rand(ids.shape, generator=g) < p`, never pick padding). Return `(inputs, labels)`: `inputs` is `ids` with the picked tokens replaced by the id `MASK`; `labels` equals the original id at picked positions and `-100` everywhere else (the loss ignores -100).

In [ ]:
def mask_tokens(ids, p, g):
    picked = None            # YOUR CODE: boolean (B, T): random < p AND not padding
    inputs = None            # YOUR CODE: ids with MASK at the picked positions
    labels = None            # YOUR CODE: original ids at picked positions, -100 elsewhere
    return inputs, labels

In [ ]:
g = torch.Generator().manual_seed(0)
batch = encode_pairs(train[:200])[0]
inp, lab = mask_tokens(batch, 0.15, g)
frac = float(((lab != -100).sum() / (batch != PAD).sum()))
check("about 15% of the real tokens are picked", 0.10 < frac < 0.20)
check("picked positions show <mask> in the input", bool((inp[lab != -100] == MASK).all()))
check("labels hold the original word", bool((lab[lab != -100] == batch[lab != -100]).all()))
check("padding is never picked and the other tokens are untouched", bool((inp[batch == PAD] == PAD).all()) and bool((inp[lab == -100] == batch[lab == -100]).all()))
print("picked fraction: %.3f" % frac)

## Experiment 1: a tiny GPT (decoder-only)
A GPT is **encoder blocks with a causal mask**, trained to predict the next token. We train a 2-block model (d = 64, 4 heads) on the English sentences of the training pairs, and compare it with two baselines: a **uniform** guess and a **bigram** model (counts of word pairs, add-0.1 smoothing). Measure on the English side of the validation pairs. Perplexity = e^(loss): "how many words is the model choosing between".

In [ ]:
class GPT(nn.Module):
    def __init__(self, vocab, d=64, h=4, dff=128, n_layers=2, p=0.1, max_len=600):
        super().__init__()
        self.d = d; self.emb = nn.Embedding(vocab, d, padding_idx=0); self.register_buffer("pe", sinusoidal_pe(max_len, d)); self.drop = nn.Dropout(p)
        self.blocks = nn.ModuleList([EncoderBlock(d, h, dff, p) for _ in range(n_layers)]); self.out = nn.Linear(d, vocab)
    def forward(self, ids):
        T = ids.shape[1]; causal = torch.tril(torch.ones(T, T, dtype=torch.bool))[None, None]
        x = self.drop(self.emb(ids) * math.sqrt(self.d) + self.pe[:T])
        for b in self.blocks: x = b(x, causal)                  # the ONLY difference from an encoder: the causal mask
        return self.out(x)
def lm_data(ps):
    seqs = [[BOS] + [tgt_v[w] for w in b.split()] + [EOS] for a, b in ps]; x = pad_batch(seqs); return x[:, :-1], x[:, 1:]
Xtr, Ytr = lm_data(train); Xva, Yva = lm_data(valid)
def train_gpt(seed=0, epochs=25, lr=2e-3, bs=32):
    torch.manual_seed(seed); m = GPT(len(tgt_v)); opt = torch.optim.Adam(m.parameters(), lr)
    for ep in range(epochs):
        m.train(); perm = torch.randperm(len(Xtr))
        for i in range(0, len(Xtr), bs):
            b = perm[i:i + bs]; lo = m(Xtr[b])
            loss = F.cross_entropy(lo.reshape(-1, lo.shape[-1]), Ytr[b].reshape(-1), ignore_index=PAD); opt.zero_grad(); loss.backward(); opt.step()
    return m
@torch.no_grad()
def lm_eval(m, X=None, Y=None):
    X, Y = (Xva, Yva) if X is None else (X, Y); m.eval(); lo = m(X)
    loss = F.cross_entropy(lo.reshape(-1, lo.shape[-1]), Y.reshape(-1), ignore_index=PAD)
    mask = Y != PAD; acc = float(((lo.argmax(-1) == Y) & mask).sum() / mask.sum()); return float(loss), acc
t0 = time.time(); gpt = train_gpt(0); loss_g, acc_g = lm_eval(gpt)
# baselines on the same validation tokens
V_ = len(tgt_v); cnt = torch.full((V_, V_), 0.1)
for a, b in zip(Xtr.reshape(-1).tolist(), Ytr.reshape(-1).tolist()):
    if a != PAD and b != PAD: cnt[a, b] += 1
bigram = cnt / cnt.sum(1, keepdim=True); mask = Yva != PAD
loss_bi = float(-torch.log(bigram[Xva[mask], Yva[mask]]).mean()); acc_bi = float((bigram[Xva[mask]].argmax(-1) == Yva[mask]).float().mean())
res = {"uniform": {"ppl": float(V_ - 0), "acc": 1 / V_}, "bigram": {"ppl": math.exp(loss_bi), "acc": acc_bi}, "tiny GPT": {"ppl": math.exp(loss_g), "acc": acc_g}}
for k, v in res.items(): print(f"{k:9s} perplexity {v['ppl']:6.2f}   next-word accuracy {v['acc']:.3f}")
print("parameters of the tiny GPT:", sum(p.numel() for p in gpt.parameters()), " (%.0f s)" % (time.time() - t0)); R["lm"] = res; R["gpt_params"] = sum(p.numel() for p in gpt.parameters())

### Generation and temperature
To generate, sample the next word from `softmax(logits / T)`. **Low T** → almost always the most likely word; **high T** → flatter, riskier. We generate 200 sentences at each temperature and measure (a) how many are **valid** (every clause is one of the corpus's simple English sentences) and (b) how many are **different** from each other.

In [ ]:
simple_en = {b[0].lower() + b[1:] for a, b in pairs if " and " not in b}
def is_valid(s):
    return all(c.strip()[:1].lower() + c.strip()[1:] in simple_en for c in s.split(" and "))
@torch.no_grad()
def sample(m, n, T, seed):
    g = torch.Generator().manual_seed(seed); m.eval(); ys = torch.full((n, 1), BOS)
    for _ in range(26):
        lo = m(ys)[:, -1] / T; nxt = torch.multinomial(F.softmax(lo, -1), 1, generator=g); ys = torch.cat([ys, nxt], 1)
    outs = []
    for row in ys[:, 1:].tolist():
        ws = []
        for t in row:
            if t == EOS: break
            ws.append(inv_t[t])
        outs.append(" ".join(ws))
    return outs
temps = {}
for T in (0.3, 1.0, 1.5, 2.5):
    outs = sample(gpt, 200, T, 0); temps[T] = {"valid": sum(map(is_valid, outs)) / 200, "unique": len(set(outs)) / 200, "example": outs[0]}
    print(f"T={T}: valid {temps[T]['valid']:.2f}  unique {temps[T]['unique']:.2f}   e.g. {outs[0]!r}")
R["temps"] = {str(k): v for k, v in temps.items()}

## Experiment 2: pre-train, then fine-tune (T5/BERT idea, measured)
**Pre-training.** A 2-block encoder (the encoder part of our translator) learns Urdu from about 8,000 Urdu sentences **without any English**: mask 15% of the words and predict them.

**Fine-tuning.** Then we train the full translator on only **300 labelled pairs**, once starting from the pre-trained encoder and once from scratch. Same everything else. Validation: the 278 held-out pairs. 3 seeds each.

In [ ]:
def causal_mask(T): return torch.tril(torch.ones(T, T, dtype=torch.bool))
class DecoderBlock(nn.Module):
    def __init__(self, d, h, d_ff, p=0.1):
        super().__init__()
        self.self_attn, self.cross_attn, self.ffn = MultiHeadAttention(d, h), MultiHeadAttention(d, h), FeedForward(d, d_ff)
        self.ln1, self.ln2, self.ln3, self.drop = nn.LayerNorm(d), nn.LayerNorm(d), nn.LayerNorm(d), nn.Dropout(p)
    def forward(self, y, memory, tgt_mask, src_mask):
        y = self.ln1(y + self.drop(self.self_attn(y, y, tgt_mask)[0]))
        a, w = self.cross_attn(y, memory, src_mask); y = self.ln2(y + self.drop(a))
        return self.ln3(y + self.drop(self.ffn(y))), w
class Seq2Seq(nn.Module):
    def __init__(self, sv, tv, d=64, h=4, dff=128, n_layers=2, p=0.1, max_len=64):
        super().__init__(); self.d = d
        self.src_emb, self.tgt_emb = nn.Embedding(sv, d, padding_idx=0), nn.Embedding(tv, d, padding_idx=0)
        self.register_buffer("pe", sinusoidal_pe(max_len, d)); self.drop = nn.Dropout(p)
        self.enc = nn.ModuleList([EncoderBlock(d, h, dff, p) for _ in range(n_layers)]); self.dec = nn.ModuleList([DecoderBlock(d, h, dff, p) for _ in range(n_layers)]); self.out = nn.Linear(d, tv)
    def encode(self, src):
        sm = (src != PAD)[:, None, None, :]; x = self.drop(self.src_emb(src) * math.sqrt(self.d) + self.pe[: src.shape[1]])
        for b in self.enc: x = b(x, sm)
        return x, sm
    def decode(self, tin, memory, sm):
        T = tin.shape[1]; tm = causal_mask(T)[None, None] & (tin != PAD)[:, None, None, :]
        y = self.drop(self.tgt_emb(tin) * math.sqrt(self.d) + self.pe[:T])
        for b in self.dec: y, _ = b(y, memory, tm, sm)
        return self.out(y)
    def forward(self, src, tin): m, sm = self.encode(src); return self.decode(tin, m, sm)
    @torch.no_grad()
    def greedy(self, src, max_len=30):
        self.eval(); memory, sm = self.encode(src); ys = torch.full((src.shape[0], 1), BOS)
        for _ in range(max_len):
            ys = torch.cat([ys, self.decode(ys, memory, sm)[:, -1].argmax(-1, keepdim=True)], 1)
            if bool((ys == EOS).any(1).all()): break
        return ys[:, 1:]
def fit_seq2seq(model, ps, epochs, seed, lr=1e-3, bs=32, warm=50):
    torch.manual_seed(seed); opt = torch.optim.Adam(model.parameters(), lr, betas=(0.9, 0.98), eps=1e-9)
    S, TI, TO = encode_pairs(ps); n = len(ps); total = epochs * math.ceil(n / bs); step = 0
    for ep in range(epochs):
        model.train(); perm = torch.randperm(n)
        for i in range(0, n, bs):
            b = perm[i:i + bs]; step += 1
            cur = lr * step / warm if step <= warm else lr * 0.5 * (1 + math.cos(math.pi * min(1, (step - warm) / max(1, total - warm))))
            for gp in opt.param_groups: gp["lr"] = cur
            lo = model(S[b], TI[b]); loss = F.cross_entropy(lo.reshape(-1, lo.shape[-1]), TO[b].reshape(-1), ignore_index=PAD); opt.zero_grad(); loss.backward(); opt.step()
    return model
def s2s_score(model, ps):
    S = encode_pairs(ps)[0]; hyps = []
    for i in range(0, len(ps), 128):
        for row in model.greedy(S[i:i + 128]).tolist():
            ws = []
            for t in row:
                if t == EOS: break
                ws.append(inv_t[t])
            hyps.append(" ".join(ws))
    refs = [b for a, b in ps]; return sacrebleu.corpus_bleu(hyps, [refs]).score, sum(h == r for h, r in zip(hyps, refs)) / len(ps)
print("ok")

In [ ]:
# --- pre-training: masked-word prediction on Urdu only ---
class MLMEncoder(nn.Module):
    def __init__(self, sv, d=64, h=4, dff=128, n_layers=2, p=0.1):
        super().__init__(); self.d = d; self.src_emb = nn.Embedding(sv, d, padding_idx=0); self.register_buffer("pe", sinusoidal_pe(64, d)); self.drop = nn.Dropout(p)
        self.enc = nn.ModuleList([EncoderBlock(d, h, dff, p) for _ in range(n_layers)]); self.head = nn.Linear(d, sv)
    def forward(self, src):
        sm = (src != PAD)[:, None, None, :]; x = self.drop(self.src_emb(src) * math.sqrt(self.d) + self.pe[: src.shape[1]])
        for b in self.enc: x = b(x, sm)
        return self.head(x)
urdu_ids = pad_batch([[src_v[w] for w in a.split()] for a, b in big[:6000]])
urdu_val = encode_pairs(valid)[0]
torch.manual_seed(0); mlm = MLMEncoder(len(src_v)); opt = torch.optim.Adam(mlm.parameters(), 1e-3); g = torch.Generator().manual_seed(1); t0 = time.time()
for ep in range(12):
    mlm.train(); perm = torch.randperm(len(urdu_ids))
    for i in range(0, len(perm), 64):
        inp, lab = mask_tokens(urdu_ids[perm[i:i + 64]], 0.15, g); lo = mlm(inp)
        loss = F.cross_entropy(lo.reshape(-1, lo.shape[-1]), lab.reshape(-1), ignore_index=-100); opt.zero_grad(); loss.backward(); opt.step()
mlm.eval()
with torch.no_grad():
    inp, lab = mask_tokens(urdu_val, 0.15, torch.Generator().manual_seed(5)); pr = mlm(inp).argmax(-1); mk = lab != -100
    mlm_acc = float((pr[mk] == lab[mk]).float().mean())
print("masked-word accuracy on held-out Urdu: %.3f  (guessing the most common word would give about %.3f)  [%.0f s]" % (mlm_acc, 1 / len(src_v), time.time() - t0))
R["mlm_acc"] = mlm_acc

In [ ]:
N_FT = 300; ft = {"from scratch": [], "pre-trained encoder": []}; t0 = time.time()
for s in (0, 1, 2):
    for name in ft:
        torch.manual_seed(s); m = Seq2Seq(len(src_v), len(tgt_v))
        if name == "pre-trained encoder":
            m.src_emb.load_state_dict(mlm.src_emb.state_dict()); m.enc.load_state_dict(mlm.enc.state_dict())
        fit_seq2seq(m, train[:N_FT], epochs=60, seed=s); b, e = s2s_score(m, valid); ft[name].append({"bleu": b, "em": e})
        print(f"seed {s} {name:20s} BLEU {b:5.1f}  exact {e:.3f}")
print("took %.0f s" % (time.time() - t0)); R["ft"] = ft; R["n_ft"] = N_FT

### Task 3: Read the transfer result

Set the variables from the numbers above (mean exact match over the 3 seeds).

In [ ]:
mean = lambda a: sum(a) / len(a)
em_scratch = None         # mean exact match, from scratch
em_pretrained = None      # mean exact match, pre-trained encoder
pretraining_helped = None # True if the pre-trained mean is higher

In [ ]:
check("scratch", abs(em_scratch - sum(r["em"] for r in ft["from scratch"]) / 3) < 1e-9)
check("pretrained", abs(em_pretrained - sum(r["em"] for r in ft["pre-trained encoder"]) / 3) < 1e-9)
check("verdict", pretraining_helped == (em_pretrained > em_scratch))
print("from scratch %.3f   pre-trained %.3f   helped: %s" % (em_scratch, em_pretrained, pretraining_helped))

## Experiment 3: attention costs n²
Self-attention builds an n × n table. We time a single head (d = 64) on random inputs for growing n and fit a straight line on a log-log plot: the slope is the exponent (2 means time ∝ n²).

### Task 4: Fit the exponent

Given the lists `ns` and `ms` (milliseconds), compute `slope` with `np.polyfit(np.log(ns), np.log(ms), 1)[0]` and the memory of ONE score matrix in megabytes (float32 = 4 bytes) for each n in `mem_mb`.

In [ ]:
ns = [256, 512, 1024, 2048, 4096]; ms = []
for n in ns:
    Q = torch.randn(1, n, 64); K = torch.randn(1, n, 64); V = torch.randn(1, n, 64)
    attention(Q, K, V)                                   # warm-up
    t0 = time.perf_counter()
    for _ in range(3): attention(Q, K, V)
    ms.append((time.perf_counter() - t0) / 3 * 1000)
slope = None       # YOUR CODE: exponent of time versus n
mem_mb = None      # YOUR CODE: list of n*n*4 / 1e6 for each n

In [ ]:
for n, t, m in zip(ns, ms, mem_mb): print(f"n={n:5d}   time {t:8.1f} ms   score matrix {m:7.1f} MB")
print("fitted exponent: %.2f" % slope)
check("memory of the score matrix is exactly n*n*4 bytes", abs(mem_mb[-1] - 4096 * 4096 * 4 / 1e6) < 1e-6)
check("the exponent is clearly above 1", slope > 1.3)

## Experiment 4: the KV cache
When a GPT generates word number *t*, it recomputes the keys and values of ALL earlier words, although they have not changed. The **KV cache** stores them and only computes the NEW word's query, key and value. Task 5 writes the cached attention step.

### Task 5: Cached attention step

Complete `step(x_new, past)`: `x_new` has shape `(B, 1, d)` (only the newest word). `past` is `None` or a tuple `(K_old, V_old)` with shape `(B, h, t, d_k)`. Compute `q, k, v` for the new word (use `split_heads` with the layer's `W_q`, `W_k`, `W_v`), **append** `k` and `v` to the old ones along dimension 2, attend `q` over ALL keys (no mask needed: the new word may see every earlier word), and return `(output, (K, V))`.

In [ ]:
class CachedMHA(MultiHeadAttention):
    def step(self, x_new, past=None):
        q = split_heads(self.W_q(x_new), self.h)
        k = None             # YOUR CODE: split_heads(self.W_k(x_new), self.h)
        v = None             # YOUR CODE
        if past is not None:
            k = None         # YOUR CODE: concatenate past[0] and k along dim=2
            v = None         # YOUR CODE: concatenate past[1] and v along dim=2
        heads, w = attention(q, k, v)
        return self.W_o(merge_heads(heads)), (k, v)

In [ ]:
torch.manual_seed(0); cm = CachedMHA(16, 4); x = torch.randn(1, 6, 16)
full, _ = cm(x, x, causal_mask(6)[None, None])                      # all 6 words at once, with the causal mask
past, outs = None, []
for t in range(6):
    o, past = cm.step(x[:, t:t + 1], past); outs.append(o)
cached = torch.cat(outs, 1)
d = float((cached - full).abs().max())
check("the cache holds 6 keys after 6 steps", past[0].shape[2] == 6)
check("step-by-step output equals the full masked computation", d < 1e-5, "k and v must be appended, and the query is only the NEW word")
print("max difference cached vs full: %.1e" % d)

In [ ]:
class CachedGPT(nn.Module):
    """The trained tiny GPT, re-using its weights, with a generation loop that can run with or without a KV cache."""
    def __init__(self, base):
        super().__init__(); self.base = base
        self.attn = nn.ModuleList()
        for b in base.blocks:
            c = CachedMHA(base.d, b.attn.h); c.load_state_dict(b.attn.state_dict()); self.attn.append(c)
    @torch.no_grad()
    def generate(self, n_new, cache=True, prompt=None):
        base = self.base; base.eval(); ys = torch.tensor([[BOS]]) if prompt is None else prompt; pasts = [None] * len(base.blocks)
        for t in range(n_new):
            if cache:
                x_in = ys[:, -1:]; pos = ys.shape[1] - 1
                x = base.emb(x_in) * math.sqrt(base.d) + base.pe[pos:pos + 1]
                for i, b in enumerate(base.blocks):
                    a, pasts[i] = self.attn[i].step(x, pasts[i]); x = b.ln1(x + a); x = b.ln2(x + b.ffn(x))
                logits = base.out(x)[:, -1]
            else:
                logits = base(ys)[:, -1]
            ys = torch.cat([ys, logits.argmax(-1, keepdim=True)], 1)
        return ys
cg = CachedGPT(gpt)
same = torch.equal(cg.generate(40, cache=True), cg.generate(40, cache=False))
print("greedy generation of 40 tokens identical with and without the cache:", same)
kv = {}
for n_new in (50, 100, 200, 400):
    t0 = time.perf_counter(); cg.generate(n_new, cache=False); t_no = time.perf_counter() - t0
    t0 = time.perf_counter(); cg.generate(n_new, cache=True); t_c = time.perf_counter() - t0
    kv[n_new] = {"no_cache_s": t_no, "cache_s": t_c, "speedup": t_no / t_c}
    print(f"generate {n_new:4d} tokens: no cache {t_no:6.2f} s   cache {t_c:6.2f} s   speedup {t_no / t_c:4.1f}x")
R["kv"] = {str(k): v for k, v in kv.items()}; R["kv_same"] = bool(same)

## Experiment 5: int8 quantisation
Store each weight matrix as 8-bit integers plus ONE float scale instead of 32-bit floats (Page 15). Symmetric per-tensor scheme: `scale = max|w| / 127`, `q = round(w / scale)` clipped to [-127, 127].

### Task 6: Quantise and dequantise

Write `quantize_int8(w)` returning `(q, scale)` with `q` an `int8` tensor, and `dequantize(q, scale)` returning a float tensor.

In [ ]:
def quantize_int8(w):
    scale = None      # YOUR CODE: w.abs().max() / 127
    q = None          # YOUR CODE: rounded, clipped to [-127, 127], converted with .to(torch.int8)
    return q, scale

def dequantize(q, scale):
    return None       # YOUR CODE

In [ ]:
w = torch.randn(64, 64)
q, s = quantize_int8(w)
check("int8 storage", q.dtype == torch.int8 and int(q.abs().max()) <= 127)
err = float((dequantize(q, s) - w).abs().max())
check("round-trip error at most half a step (scale / 2)", err <= float(s) / 2 + 1e-6)
print("max round-trip error %.4f, step size %.4f" % (err, float(s)))

In [ ]:
import copy
qgpt = copy.deepcopy(gpt); fp32_bytes = int8_bytes = 0
with torch.no_grad():
    for name, p in qgpt.named_parameters():
        if p.dim() == 2 and "emb" not in name:                     # all weight matrices; keep embeddings, biases and norms in float
            q, s = quantize_int8(p.data); p.data.copy_(dequantize(q, s)); fp32_bytes += p.numel() * 4; int8_bytes += p.numel() + 4
loss_q, acc_q = lm_eval(qgpt)
quant = {"ppl_fp32": math.exp(loss_g), "ppl_int8": math.exp(loss_q), "acc_fp32": acc_g, "acc_int8": acc_q, "bytes_fp32": fp32_bytes, "bytes_int8": int8_bytes}
print("quantised weight matrices: %d bytes -> %d bytes (%.1fx smaller)" % (fp32_bytes, int8_bytes, fp32_bytes / int8_bytes))
print("perplexity %.3f -> %.3f     next-word accuracy %.3f -> %.3f" % (math.exp(loss_g), math.exp(loss_q), acc_g, acc_q))
R["quant"] = quant

## Wrap-up
- Same blocks, different masks: BERT sees everything, GPT sees the past, T5 does both with cross-attention.
- Temperature trades validity for variety.
- Pre-training on unlabelled text helped (or did not, see your numbers) when labelled data is scarce.
- Attention time and memory grow with n²; the KV cache removes repeated work at the price of memory; int8 quantisation shrinks weights about 4×.

**Think about it:** which of these measurements would change if we used a 1,000× larger model and real text? Which would stay the same?

In [ ]:
print("RESULT_JSON " + json.dumps(R))